# Практическое занятие 12

## Кластеризация событий кибербезопасности методом k-means и выявление аномалий

Ноутбук рассчитан на 80 минут. Модель обучается без целевой переменной. Столбцы `asset_role`,
`is_attack` и `attack_type` используются только после кластеризации — для учебной интерпретации
и контрольной проверки.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (silhouette_score,
                             calinski_harabasz_score,
                             davies_bouldin_score)

RANDOM_STATE = 42
FEATURES = ["connections_per_min",
            "unique_destinations",
            "failed_logins",
            "outbound_mb",
            "session_duration_sec"]

# Если файл загружен в другую папку, измените путь.
DATA_PATH = "cybersecurity_kmeans_practice_dataset.csv"
df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])
df.head()

## Задание 1. Аудит и подготовка данных

1. Проверьте размер, пропуски, полные дубликаты и состав частей `train`, `calibration`, `monitoring`.
2. Объясните назначение каждой части.
3. Обучите `StandardScaler` только на `train`, затем тем же объектом преобразуйте `calibration` и `monitoring`.
4. Объясните, почему метки атак и роли узлов нельзя включать в матрицу признаков.

In [ ]:
# выполните аудит датасета и сформируйте train_df, calibration_df, monitoring_df.
# обучите StandardScaler только на train и получите X_train,
#       X_calibration, X_monitoring.

## Задание 2. Выбор числа кластеров

Для каждого значения $k$ от 2 до 8 обучите отдельную модель `KMeans` с
`init="k-means++"`, `n_init=20`, `random_state=42`. Рассчитайте:

- Inertia — меньше лучше, но показатель монотонно уменьшается при росте $k$;
- коэффициент силуэта — больше лучше;
- индекс Калински—Харабаша — больше лучше;
- индекс Дэвиса—Болдина — меньше лучше.

Постройте четыре графика средствами `matplotlib`. Определите «локоть» и выберите $k$ по
совокупности графиков и содержательному смыслу профилей.

In [ ]:
# рассчитайте четыре внутренние метрики при k=2,...,8.
# выведите metrics_df и постройте четыре графика matplotlib.
# запишите выбранное k и краткое обоснование.

## Задание 3. k-means, k-means++ и интерпретация центроидов

1. Обучите две модели при выбранном $k$: с `init="random"` и `init="k-means++"`.
2. Сопоставьте `inertia_` и `n_iter_`. Объясните роль `n_init` и `random_state`.
3. Переведите центроиды k-means++ в исходные единицы методом `inverse_transform`.
4. Сопоставьте кластеры с контрольным полем `asset_role` только после обучения.
5. Постройте две проекции кластеров и центроидов. Помните, что модель использует все пять признаков.

In [ ]:
# обучите kmeans_random и kmeans_plus при k=3.
# сравните inertia_ и n_iter_, восстановите центроиды в исходных единицах,
#       сформируйте таблицу cluster × asset_role и постройте две проекции.

## Задание 4. Выявление аномалий в двух режимах

Оценка аномальности объекта — расстояние до назначенного центроида:

$$A(x_i)=\lVert x_i-\mu_{c_i}\rVert_2.$$

Для каждого кластера используйте порог 99-го процентиля.

**Режим А — ретроспективный анализ всей выборки.** Масштабируйте всю выборку, обучите k-means
на ней, вычислите расстояния методом `transform` и отметьте наиболее удалённые объекты.

**Режим Б — новые объекты.** Не переобучайте модель. Получите пороги на нормальной
`calibration`, затем методами `predict` и `transform` оцените `monitoring`. Сопоставьте результат
с `is_attack` только после получения прогнозов.

In [ ]:
# обучите scaler_all и kmeans_all на всей выборке, рассчитайте score и
#         отдельный 99%-порог внутри каждого кластера.
# по calibration рассчитайте кластерные пороги для ранее обученной kmeans_plus,
#         затем оцените monitoring без переобучения.
# выведите число срабатываний и результаты по attack_type.

## Задание 5. Визуализация и записка SOC

1. Постройте распределение оценок `monitoring` для нормы и атак с помощью `matplotlib`.
2. Отметьте кандидатов на аномалии на одной двумерной проекции.
3. Составьте записку SOC из 5–7 предложений: объём и части данных, выбранное $k$, способ
инициализации, правило порога, число срабатываний, приоритетные события и ограничения.

Ключевое ограничение: большое расстояние означает нетипичность относительно найденного профиля,
но не доказывает атаку.

In [ ]:
# постройте гистограммы оценок и проекцию с кандидатами на аномалии.
# напишите записку SOC в следующей Markdown-ячейке.

### Записка SOC

*Введите 5–7 предложений.*